# Chapter 8 -- 神经网络基础

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chaosfrey-arch/news-classification-tutorial/blob/main/chapter08_neural_networks.ipynb)

**本章目标：** 理解神经网络是什么，训练的本质是什么。

**预计时间：** 60 分钟

> 强烈推荐：[3Blue1Brown 神经网络可视化系列](https://www.3blue1brown.com/topics/neural-networks)（即使不懂英文，动画也能帮你理解）

## 8.1 为什么需要神经网络？

TF-IDF + 朴素贝叶斯能做到约 90% 的准确率，已经不错了。

但它们的根本局限：
- TF-IDF 不理解语义："football" 和 "soccer" 是同一件事，但被当成完全不同的词
- 它不理解语序："dog bites man" 和 "man bites dog" 意思截然不同，但词袋看起来一样

**神经网络** 能学习更复杂的模式，突破这些限制。

## 8.2 神经元（Neuron）是什么？

人工神经元模仿生物神经元：

```
输入 x1, x2, x3
     ↓  ↓  ↓
   × w1 × w2 × w3    ← 权重（weights），是模型「学习」的东西
     ↓
  求和 + 偏置 b      ← z = w1*x1 + w2*x2 + w3*x3 + b
     ↓
  激活函数 f(z)      ← 引入非线性
     ↓
   输出 y
```

**权重（Weights）** 是模型里唯一需要学习的参数。训练的本质就是调整权重，让预测越来越准。

In [ ]:
# 用纯 Python 手写一个神经元，感受一下
import numpy as np

# 一个有 3 个输入的神经元
def neuron(x, w, b):
    """
    x: 输入向量 [x1, x2, x3]
    w: 权重向量 [w1, w2, w3]
    b: 偏置（bias）
    """
    z = np.dot(x, w) + b  # 加权求和 + 偏置
    output = max(0, z)     # ReLU 激活函数：负数变 0，正数保留
    return output

# 测试
x = np.array([0.5, 0.3, 0.8])  # 假设的输入（3 个 TF-IDF 特征）
w = np.array([0.2, -0.4, 0.7]) # 权重（随机初始化）
b = 0.1                          # 偏置

result = neuron(x, w, b)
print(f'输入：{x}')
print(f'权重：{w}')
print(f'加权求和 + 偏置：{np.dot(x,w)+b:.4f}')
print(f'ReLU 激活后输出：{result:.4f}')

## 8.3 激活函数（Activation Function）

**为什么需要激活函数？**
如果没有激活函数，多层神经元叠在一起等于一层（因为线性函数的组合还是线性函数）。激活函数引入非线性，让网络能学习复杂模式。

**常用激活函数：**

- **ReLU（Rectified Linear Unit）：** `f(x) = max(0, x)`
  - 负数变 0，正数保留
  - 计算快，效果好，深度学习最常用

- **Softmax：** 把数字变成概率分布（所有类别概率加起来 = 1）
  - 用于最后一层输出层
  - 例：输出 [2.1, 0.3, 5.2, 1.1] → Softmax → [0.07, 0.01, 0.89, 0.03]（概率）

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(-4, 4, 200)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# ReLU
relu = np.maximum(0, x)
axes[0].plot(x, relu, 'steelblue', linewidth=2.5)
axes[0].axhline(0, color='gray', linewidth=0.8)
axes[0].axvline(0, color='gray', linewidth=0.8)
axes[0].set_title('ReLU: f(x) = max(0, x)', fontsize=13)
axes[0].set_xlabel('x'); axes[0].set_ylabel('f(x)')
axes[0].grid(True, alpha=0.3)
axes[0].annotate('负数变0', xy=(-2, 0), xytext=(-3.5, 1),
                 arrowprops=dict(arrowstyle='->', color='red'), color='red', fontsize=10)

# Softmax 示例
logits = np.array([2.1, 0.3, 5.2, 1.1])
exp_logits = np.exp(logits)
softmax = exp_logits / exp_logits.sum()
axes[1].bar(['World', 'Business', 'Sports', 'Science/Tech'],
            softmax, color=['steelblue','coral','mediumseagreen','mediumpurple'])
axes[1].set_title('Softmax Output (sum=1.00)', fontsize=13)
axes[1].set_ylabel('Probability')
for i, v in enumerate(softmax):
    axes[1].text(i, v+0.01, f'{v:.2f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## 8.4 前向传播（Forward Propagation）

数据从输入层流向输出层的过程：

```
输入层（文章特征）
    ↓
隐藏层 1（Dense 128 neurons）: 每个神经元 = 加权求和 + ReLU
    ↓
隐藏层 2（Dense 64 neurons）
    ↓
输出层（Dense 4 neurons）: Softmax → 4 个类别的概率
```

最后概率最大的类别就是预测结果。

## 8.5 损失函数（Loss Function）

**损失函数** 衡量预测有多错。

分类问题常用 **Cross-Entropy Loss（交叉熵损失）**：
- 如果真实类别是 Sports，模型预测 Sports 概率 = 0.9 → Loss 小（好）
- 如果真实类别是 Sports，模型预测 Sports 概率 = 0.1 → Loss 大（差）

训练的目标：**最小化 Loss。**

## 8.6 反向传播（Backpropagation）+ 梯度下降

**梯度下降（Gradient Descent）直觉：**
想象你在山上，目标是到达山谷（Loss 最小点）：
- 站在当前位置，看哪个方向向下最陡
- 往那个方向走一小步
- 重复，直到到达谷底

**反向传播：** 自动计算「每个权重对 Loss 的影响」，告诉梯度下降该往哪个方向调整权重。

你不需要手写这些计算，Keras 自动完成。

> 深入学习：[Stanford CS231n 反向传播讲义](https://cs231n.github.io/optimization-2/)

## 8.7 关键超参数

| 参数 | 含义 | 类比 |
|------|------|------|
| **Epoch** | 把整个训练集看一遍 = 1 个 epoch | 把练习题做一遍 |
| **Batch Size** | 每次更新权重用多少条数据 | 每做 256 道题对一次答案 |
| **Learning Rate** | 每步走多大 | 步子大小 |
| **Validation Split** | 从训练集留出一部分做验证 | 用部分练习题自测 |

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
import pandas as pd, numpy as np, re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def clean_text(t):
    t = str(t).lower()
    t = re.sub(r'http\S+|www\S+|https\S+', '', t)
    t = re.sub(r'[^a-z\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

df = pd.read_csv('dataset.csv').dropna(subset=['Class']).copy()
df['Description'] = df['Description'].fillna('')
df['text'] = (df['Title'] + ' ' + df['Description']).apply(clean_text)
le = LabelEncoder()
y = le.fit_transform(df['Class'])
X_train, X_test, y_train, y_test = train_test_split(
    df['text'].values, y, test_size=0.3, random_state=42, stratify=y)

tfidf = TfidfVectorizer(max_features=5000, sublinear_tf=True, min_df=2)
X_tr = tfidf.fit_transform(X_train).toarray()  # Dense 层需要稠密矩阵
X_te = tfidf.transform(X_test).toarray()
y_tr_cat = to_categorical(y_train, 4)
y_te_cat = to_categorical(y_test, 4)
print('数据准备完毕！')

In [ ]:
# 搭建一个简单的全连接神经网络，感受 Keras 的使用方式
model = Sequential([
    Dense(256, activation='relu', input_shape=(5000,)),  # 隐藏层 1
    Dropout(0.3),                                         # Dropout：防止过拟合
    Dense(128, activation='relu'),                        # 隐藏层 2
    Dropout(0.3),
    Dense(4, activation='softmax')                        # 输出层：4 个类别
], name='SimpleNN')

# compile：指定优化器、损失函数、评估指标
model.compile(
    optimizer='adam',              # Adam 是最常用的优化器，自动调整学习率
    loss='categorical_crossentropy',  # 多分类问题的标准损失函数
    metrics=['accuracy']
)

model.summary()  # 打印网络结构

In [ ]:
# EarlyStopping：当验证集 loss 连续 3 轮不下降时，自动停止训练
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    X_tr, y_tr_cat,
    epochs=20,               # 最多训练 20 轮
    batch_size=256,          # 每批 256 条数据
    validation_split=0.1,   # 训练集的 10% 用于验证
    callbacks=[early_stop],  # 早停
    verbose=1
)
print(f'实际训练了 {len(history.history["loss"])} 轮（早停生效）')

In [ ]:
# 画训练曲线
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, (metric, title) in enumerate([('accuracy','Accuracy'),('loss','Loss')]):
    axes[i].plot(history.history[metric], label='Train')
    axes[i].plot(history.history[f'val_{metric}'], label='Validation')
    axes[i].set_title(f'Simple NN -- {title}', fontsize=13)
    axes[i].set_xlabel('Epoch')
    axes[i].set_ylabel(title)
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('观察：训练曲线和验证曲线接近 = 没有过拟合。若验证 loss 上升而训练 loss 继续下降 = 过拟合！')

## 总结

| 概念 | 含义 |
|------|------|
| 神经元 | 加权求和 + 激活函数 |
| 权重（Weights）| 模型学习的参数 |
| 前向传播 | 数据从输入到输出的计算过程 |
| 损失函数 | 衡量预测有多错（越小越好）|
| 反向传播 | 计算每个权重对 Loss 的影响 |
| 梯度下降 | 沿 Loss 减小方向更新权重 |
| EarlyStopping | 验证 loss 不降时自动停止 |
| Dropout | 随机关闭神经元，防止过拟合 |

**下一章 →** Chapter 9：词向量 Embedding